In [1]:
USE_KAGGLE_SANDBOX = True
KAGGLE_SANDBOX_KWARGS = {
    'dataset_data_sources': ['greynearing/camels-us-531-era'],
    'enable_gpu': True,
    'enable_internet': False,
    'machine_shape': 'NvidiaTeslaT4Highmem',
}
C_PUCT = 2.5

# CAMELS-US ERA — PUR (18-Fold HUC-02) Minimal Guidance

## Objective
Maximize the **out-of-sample Median Nash-Sutcliffe Efficiency (NSE)** across all **531 CAMELS-US basins** under the **18-Fold Leave-One-HUC02-Region-Out Prediction in Ungauged Regions (PUR)** cross-validation protocol on the evaluation period (`1999-10-01` to `2003-09-30`, 4 water years, `1,461` days).

## Data & Split Protocol
- **Spatial Regime (18-Fold HUC-02 PUR Cross-Validation)**:
  - `CV_FOLDS` (loaded from `pur_18fold_huc02_splits.json`) partitions the 531 basins by USGS 2-digit Hydrologic Unit Code (`HUC_02`, 18 regions).
  - In each fold `k`, an **entire HUC-02 region** (`fold['test_basin_mask']`) is held out as completely ungauged (`fold['qobs_fold_train']` is `NaN` on that region), and the model is trained on the remaining 17 HUC-02 regions (`fold['train_basin_mask']`) during `2003-10-01` to `2008-09-30`.
  - Predict out-of-sample discharge for `fold['test_idx']` and assemble the full `Q_PRED` array of shape `(531, 3653)` across all 18 regional folds.
- **Strict Newman Temporal Containment (`1998-10-01` to `2008-09-30`, `3,653` days)**:
  - `1998-10-01` to `1999-09-30` (`365` days): 1-year meteorological spin-up prior to evaluation (`QOBS_TRAIN = NaN`).
  - `1999-10-01` to `2003-09-30` (`1,461` days, `ERA_EVAL_MASK`): Out-of-sample ERA hill-climb evaluation period (`QOBS_TRAIN = NaN`).
  - `2003-10-01` to `2008-09-30` (`1,827` days, `ERA_TRAIN_MASK`): ERA model weights training period on each fold's 17 training regions.
- **Inputs Available**:
  - `DYNAMIC_INPUTS`: shape `(531, 3653, 15)` (`float32`) — 5 daily meteorological variables (`prcp`, `srad`, `tmax`, `tmin`, `vp`) across 3 forcing products (`Daymet`, `Maurer`, `NLDAS`).
  - `AUX_FORCINGS`: shape `(531, 3653, 6)` (`float32`) — daily `dayl` and `swe` across `Daymet`, `Maurer`, `NLDAS`.
  - `STATIC_ATTRIBUTES`: shape `(531, 27)` (`float32`) — 27 catchment climate, topography, soil, geology, and vegetation attributes (plus `BASIN_AREAS`, `LAT`, `LON`, `HUC_02`).
- **Required Output**:
  - The mutable cell must produce `Q_PRED` as a `float32` NumPy array of shape `(531, 3653)` containing daily streamflow predictions (`mm/day`).

## Modeling & Exploration Scope
You are free to explore any machine learning architecture, sequence length, feature representation, normalization strategy, loss function, regularization, optimizer schedule, or multi-model ensembling strategy, provided all predictions strictly respect the causal temporal masks and spatial fold boundaries.

In [2]:
import os
import time
import math
import json
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Ensure reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

_SEARCH_DIRS = [
    Path("/kaggle/input/camels-us-531-era"),
    Path("/kaggle/input/datasets/greynearing/camels-us-531-era"),
    Path(__file__).resolve().parent.parent / "splits" if "__file__" in globals() else Path("../splits"),
    Path("../splits"),
    Path("./splits"),
]

def _find_file(filename: str) -> Path:
    for d in _SEARCH_DIRS:
        candidate = d / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {filename} in any of {_SEARCH_DIRS}")

_train_npz = np.load(_find_file("camels_era_train.npz"), allow_pickle=True)
_basins_json = json.loads(_find_file("basins_531.json").read_text(encoding="utf-8"))

BASINS = _train_npz["basins"].astype(str)
_canonical_basins = [str(b) for b in _basins_json]
assert list(BASINS) == _canonical_basins, "BASINS order mismatch against canonical basins_531.json"

# Strictly enforce containment to the Newman training window (1998-10-01 to 2008-09-30, 3653 days)
_raw_dates = np.array([str(d) for d in _train_npz["dates"]])
_newman_window = (_raw_dates >= "1998-10-01") & (_raw_dates <= "2008-09-30")
DATES = _raw_dates[_newman_window]
assert len(DATES) == 3653 and DATES[0] == "1998-10-01" and DATES[-1] == "2008-09-30"

DYNAMIC_INPUTS = _train_npz["dynamic_inputs"][:, _newman_window, :].astype(np.float32)
DYNAMIC_FEATURE_NAMES = [str(x) for x in _train_npz["dynamic_feature_names"]]
AUX_FORCINGS = _train_npz["aux_forcings"][:, _newman_window, :].astype(np.float32)
AUX_FEATURE_NAMES = [str(x) for x in _train_npz["aux_feature_names"]]
STATIC_ATTRIBUTES = _train_npz["static_attributes"].astype(np.float32)
STATIC_ATTRIBUTE_NAMES = [str(x) for x in _train_npz["static_attribute_names"]]

ERA_TRAIN_MASK = (DATES >= "2003-10-01") & (DATES <= "2008-09-30")
ERA_EVAL_MASK = (DATES >= "1999-10-01") & (DATES <= "2003-09-30")
assert int(ERA_TRAIN_MASK.sum()) == 1827, f"Expected 1827 training days, got {int(ERA_TRAIN_MASK.sum())}"
assert int(ERA_EVAL_MASK.sum()) == 1461, f"Expected 1461 eval days, got {int(ERA_EVAL_MASK.sum())}"

QOBS_TRAIN = np.where(
    ERA_TRAIN_MASK[None, :],
    _train_npz["qobs_train"][:, _newman_window].astype(np.float32),
    np.nan,
).astype(np.float32)

BASIN_AREAS = _train_npz["basin_areas"].astype(np.float32)
LAT = _train_npz["lat"].astype(np.float32)
LON = _train_npz["lon"].astype(np.float32)

SPATIAL_REGIME = "pur_18fold"
_splits_json = json.loads(_find_file("pur_18fold_huc02_splits.json").read_text(encoding="utf-8"))
CV_SPLITS = [v for _, v in sorted(_splits_json["folds"].items(), key=lambda kv: int(kv[1]["fold_index"]))]
assert len(CV_SPLITS) == 18, f"Expected 18 PUR folds, got {len(CV_SPLITS)}"

_basin_to_idx = {str(b): i for i, b in enumerate(BASINS)}
CV_FOLDS = []
_covered_test = np.zeros(len(BASINS), dtype=bool)
for _fold in CV_SPLITS:
    _fid = int(_fold["fold_index"])
    _test_idx = np.array([_basin_to_idx[str(b)] for b in _fold["test_basins"]], dtype=np.int64)
    _train_idx = np.array([_basin_to_idx[str(b)] for b in _fold["train_basins"]], dtype=np.int64)
    _test_mask = np.zeros(len(BASINS), dtype=bool)
    _test_mask[_test_idx] = True
    _train_mask = np.zeros(len(BASINS), dtype=bool)
    _train_mask[_train_idx] = True
    assert not np.any(_test_mask & _train_mask), f"Train/test overlap in PUR fold {_fid}"
    assert not np.any(_covered_test & _test_mask), f"Duplicate test basins in PUR fold {_fid}"
    _covered_test |= _test_mask
    CV_FOLDS.append({
        "fold_id": _fid,
        "held_out_huc02": str(_fold["huc_02"]),
        "train_basins": [str(b) for b in _fold["train_basins"]],
        "test_basins": [str(b) for b in _fold["test_basins"]],
        "train_idx": _train_idx,
        "test_idx": _test_idx,
        "train_basin_mask": _train_mask,
        "test_basin_mask": _test_mask,
        "qobs_fold_train": np.where(
            _train_mask[:, None] & ERA_TRAIN_MASK[None, :], QOBS_TRAIN, np.nan
        ).astype(np.float32),
    })
assert np.all(_covered_test), "18 PUR folds do not cover all 531 basins!"

print(f"Loaded CAMELS-US 531 dataset ({SPATIAL_REGIME}): {len(BASINS)} basins, {len(DATES)} days ({DATES[0]} to {DATES[-1]}), {len(CV_FOLDS)} HUC-02 folds.")
print(f"ERA_TRAIN_MASK (2003-10-01..2008-09-30): {int(ERA_TRAIN_MASK.sum())} days | ERA_EVAL_MASK (1999-10-01..2003-09-30): {int(ERA_EVAL_MASK.sum())} days")
print(f"Dynamic inputs: {DYNAMIC_INPUTS.shape}, Aux: {AUX_FORCINGS.shape}, Static: {STATIC_ATTRIBUTES.shape}")
print(f"Available GPUs: {NUM_GPUS} (Primary device: {DEVICE})")

Loaded CAMELS-US 531 dataset (pur_18fold): 531 basins, 3653 days (1998-10-01 to 2008-09-30), 18 HUC-02 folds.
ERA_TRAIN_MASK (2003-10-01..2008-09-30): 1827 days | ERA_EVAL_MASK (1999-10-01..2003-09-30): 1461 days
Dynamic inputs: (531, 3653, 15), Aux: (531, 3653, 6), Static: (531, 27)
Available GPUs: 2 (Primary device: cuda)


In [3]:
# Begin Stage

In [4]:
INSTRUCTION = [
    "In this 18-fold PUR benchmark, each fold holds out an ENTIRE HUC-02 hydroclimatic region (`CV_FOLDS`). Prevent regional overfitting by replacing raw geographic coordinates (`LAT`, `LON`) or discontinuous static lookups with heavily regularized, low-dimensional static bottleneck embeddings, feature dropout, and Gaussian noise on static attributes.",
    "Explore invariant representation learning across regions: normalize dynamic forcings relative to each catchment's long-term hydroclimatic regime (e.g., rolling anomalies and ratios relative to 365-day moving averages) so the sequence model learns universal input-response dynamics that extrapolate to unseen HUC-02 regions.",
    "Design a Feature-wise Linear Modulation (FiLM) or hypernetwork conditioning layer with strong L2 weight decay, spectral/layer normalization, and static dropout (`p=0.2..0.3`) so static catchment conditioning extrapolates smoothly when an entire geographic region is held out.",
    "Engineer rich multi-timescale temporal features (exponential moving averages with half-lives from 2 to 365 days, cumulative water-year precipitation and radiation deficits, and inter-product forcing differences across Daymet, Maurer, and NLDAS) that capture catchment memory directly from meteorological time series.",
    "Explore a multi-scale sequence architecture (combining gated recurrent units/LSTMs with dilated causal convolutions or attention over 365-day history) trained with composite basin-normalized NSE + log-MSE loss and EMA weight averaging across all 18 regional folds.",
    "Use Mixup / interpolating regularization across training basins from different HUC-02 regions (or domain-adversarial / variance-penalized loss across training HUC-02 regions) so the model prioritizes features that perform consistently across all hydroclimatic regions.",
    "Utilize multi-GPU parallel fold training (`torch.cuda.device_count()`) and FP16/BF16 mixed precision so all 18 regional folds can train a multi-seed / multi-head ensemble cleanly within the time budget.",
    "Synthesize the strongest regional-extrapolation techniques (coordinate exclusion / regularized static bottleneck, regime-invariant dynamic ratios, FiLM modulation, EMA weight averaging, and multi-seed ensembling across all 18 HUC-02 folds) to maximize PUR Median NSE."
]
VALUE_REGEX = r'(?m)^Candidate metric to be maximized: (.*)$'
OBJECTIVE = 'maximize'
WORST_VALUE = -2e308

In [5]:
# [rewrite_cell]
# ==============================================================================
# OPTIMIZED DUAL-FiLM LSTM FOR GEOGRAPHIC EXTRAPOLATION (PUR)
# ==============================================================================

# 1. Enhanced Feature Engineering
def prepare_features():
    # Log-transform highly skewed physical attributes
    area_log = np.log1p(BASIN_AREAS).reshape(-1, 1)

    # Climate descriptors for regional context
    # Precip indices (Daymet, Maurer, NLDAS)
    p_mean = DYNAMIC_INPUTS[:, :, [0, 5, 10]].mean(axis=(1, 2)).reshape(-1, 1)
    p_mean_log = np.log1p(p_mean)
    # Temp indices (Tmax, Tmin across 3 products)
    t_mean = DYNAMIC_INPUTS[:, :, [2, 3, 7, 8, 12, 13]].mean(axis=(1, 2)).reshape(-1, 1)
    # Aridity Proxy: P / (T + 10)
    aridity = (p_mean / (np.maximum(t_mean, -5.0) + 10.0)).reshape(-1, 1)

    # Combine all statics: Raw + Added context
    X_s = np.column_stack([
        STATIC_ATTRIBUTES,
        LAT.reshape(-1, 1),
        LON.reshape(-1, 1),
        area_log,
        p_mean_log,
        t_mean,
        aridity
    ])

    # Seasonal encoding (Sin/Cos of day of year)
    d_idx = np.arange(len(DATES))
    sin_d = np.sin(2 * np.pi * d_idx / 365.25)
    cos_d = np.cos(2 * np.pi * d_idx / 365.25)
    seasonal = np.stack([np.tile(sin_d, (len(BASINS), 1)),
                         np.tile(cos_d, (len(BASINS), 1))], axis=-1).astype(np.float32)

    # Dynamic inputs + Aux forcings (SWE, Dayl) + Seasonal
    X_d = np.concatenate([DYNAMIC_INPUTS, AUX_FORCINGS, seasonal], axis=-1).astype(np.float32)

    # Handle NaNs in statics using global means (excluding target-related NaNs)
    col_means = np.nanmean(X_s, axis=0)
    inds = np.where(np.isnan(X_s))
    X_s[inds] = np.take(col_means, inds[1])

    return X_d, X_s

X_dyn_raw, X_stat_raw = prepare_features()

# Hyperparameters
SEQ_LEN = 365
HIDDEN_SIZE = 256
DROPOUT = 0.3
BATCH_SIZE = 128
LR_MAX = 1e-3
EPOCHS = 50
SAMPLES_PER_BASIN = 64
NUM_SEEDS = 2  # Averaging seeds for stability

class StatEncoder(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(out_dim, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU()
        )
    def forward(self, x):
        return self.net(x)

class DualFiLMLSTM(nn.Module):
    def __init__(self, dyn_dim, stat_dim, hidden_size, dropout):
        super().__init__()
        self.h = hidden_size
        self.stat_enc = StatEncoder(stat_dim, hidden_size)

        # FiLM 1: Modulation for inputs
        self.film_in = nn.Linear(hidden_size, dyn_dim * 2)
        # FiLM 2: Modulation for hidden states
        self.film_out = nn.Linear(hidden_size, hidden_size * 2)

        # Hidden state initialization
        self.init_h = nn.Linear(hidden_size, hidden_size)
        self.init_c = nn.Linear(hidden_size, hidden_size)

        self.lstm = nn.LSTM(dyn_dim, hidden_size, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x_d, x_s, return_seq=False):
        # Generate static identity embedding
        stat_emb = self.stat_enc(x_s)

        # FiLM 1: Modulate Input Forcings
        params_in = self.film_in(stat_emb).unsqueeze(1)
        gamma1, beta1 = params_in.chunk(2, dim=-1)
        x_d = x_d * (1 + gamma1) + beta1

        # Initial states derived from physics
        h0 = self.init_h(stat_emb).unsqueeze(0).contiguous()
        c0 = self.init_c(stat_emb).unsqueeze(0).contiguous()

        out, _ = self.lstm(x_d, (h0, c0))

        # FiLM 2: Modulate Hidden States (Regional Response Adjustment)
        params_out = self.film_out(stat_emb).unsqueeze(1)
        gamma2, beta2 = params_out.chunk(2, dim=-1)
        out = out * (1 + gamma2) + beta2

        if not return_seq:
            out = out[:, -1, :]
        return self.head(out).squeeze(-1)

def train_fold(fold, device_id):
    dev = torch.device(f"cuda:{device_id}")
    train_idx, test_idx = fold["train_idx"], fold["test_idx"]
    train_mask = fold["train_basin_mask"]

    # 1. Dynamic Normalization (Fold-specific)
    xd_m = X_dyn_raw[train_mask][:, ERA_TRAIN_MASK].mean(axis=(0, 1), keepdims=True)
    xd_s = X_dyn_raw[train_mask][:, ERA_TRAIN_MASK].std(axis=(0, 1), keepdims=True) + 1e-6
    X_d_gpu = torch.from_numpy((X_dyn_raw - xd_m) / xd_s).to(dev)

    # 2. Static Normalization (Fold-specific)
    xs_m = X_stat_raw[train_mask].mean(axis=0, keepdims=True)
    xs_s = X_stat_raw[train_mask].std(axis=0, keepdims=True) + 1e-6
    X_s_gpu = torch.from_numpy((X_stat_raw - xs_m) / xs_s).to(dev)

    # 3. Target Transformation (Sqrt)
    q_target = np.sqrt(np.maximum(fold["qobs_fold_train"], 0.0))
    y_m, y_s = np.nanmean(q_target), np.nanstd(q_target) + 1e-6
    Y_gpu = torch.from_numpy((q_target - y_m) / y_s).to(dev)

    # Valid time indices for training (ensuring spin-up)
    valid_t = {b: np.where(ERA_TRAIN_MASK & (~np.isnan(q_target[b])))[0] for b in train_idx}
    valid_t = {b: t[t >= SEQ_LEN-1] for b, t in valid_t.items() if len(t) > 0}

    fold_preds = np.zeros((len(test_idx), len(DATES)), dtype=np.float32)
    seq_offs = torch.arange(-SEQ_LEN + 1, 1, device=dev).unsqueeze(0)

    for seed in range(NUM_SEEDS):
        torch.manual_seed(SEED + seed)
        model = DualFiLMLSTM(X_d_gpu.shape[-1], X_s_gpu.shape[-1], HIDDEN_SIZE, DROPOUT).to(dev)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR_MAX, weight_decay=0.05)

        steps = math.ceil((len(valid_t) * SAMPLES_PER_BASIN) / BATCH_SIZE)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR_MAX,
                                                        steps_per_epoch=steps, epochs=EPOCHS)
        criterion = nn.MSELoss()
        scaler = torch.amp.GradScaler("cuda")
        rng = np.random.RandomState(SEED + seed)

        for epoch in range(EPOCHS):
            model.train()
            b_list, t_list = [], []
            for b, t_arr in valid_t.items():
                idx = rng.choice(t_arr, size=SAMPLES_PER_BASIN, replace=(len(t_arr) < SAMPLES_PER_BASIN))
                b_list.append(np.full(len(idx), b))
                t_list.append(idx)

            b_all, t_all = np.concatenate(b_list), np.concatenate(t_list)
            p = rng.permutation(len(b_all))
            b_all, t_all = b_all[p], t_all[p]

            for i in range(0, len(b_all), BATCH_SIZE):
                bb = torch.from_numpy(b_all[i:i+BATCH_SIZE]).to(dev)
                bt = torch.from_numpy(t_all[i:i+BATCH_SIZE]).to(dev)

                optimizer.zero_grad()
                with torch.amp.autocast("cuda"):
                    # Regularization: Jitter static inputs during training to prevent memorization
                    s_feat = X_s_gpu[bb]
                    if model.training:
                        s_feat = s_feat + torch.randn_like(s_feat) * 0.015

                    x_d_seq = X_d_gpu[bb.unsqueeze(1), bt.unsqueeze(1) + seq_offs]
                    y_hat = model(x_d_seq, s_feat)
                    loss = criterion(y_hat, Y_gpu[bb, bt])

                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

        # Inference: Full sequence (including 1-year spin-up)
        model.eval()
        with torch.no_grad(), torch.amp.autocast("cuda"):
            test_idx_gpu = torch.from_numpy(test_idx).to(dev)
            # Predict full series [0...3652]
            p_norm = model(X_d_gpu[test_idx_gpu], X_s_gpu[test_idx_gpu], return_seq=True)
            # De-normalize and Inverse Transform (Square)
            p_mm = (torch.clamp(p_norm * y_s + y_m, min=0.0)**2).cpu().numpy()
            fold_preds += p_mm / NUM_SEEDS

    return test_idx, fold_preds

# Run 18 PUR Folds in parallel across available GPUs
Q_PRED = np.zeros((len(BASINS), len(DATES)), dtype=np.float32)
t_start = time.time()
with ThreadPoolExecutor(max_workers=max(1, NUM_GPUS)) as executor:
    futures = [executor.submit(train_fold, f, i % max(1, NUM_GPUS)) for i, f in enumerate(CV_FOLDS)]
    for fut in futures:
        idx, preds = fut.result()
        Q_PRED[idx] = preds

print(f"PUR Dual-FiLM-LSTM Finished. Time: {time.time()-t_start:.1f}s")

PUR Dual-FiLM-LSTM Finished. Time: 4835.2s


In [6]:
# ==============================================================================
# Immutable ERA Hill-Climb Evaluation (Strictly 1999-10-01 to 2003-09-30)
# ==============================================================================
_eval_npz = np.load(_find_file("era_eval_targets.npz"), allow_pickle=True)
_eval_raw_dates = np.array([str(d) for d in _eval_npz["dates"]])
_eval_window = (_eval_raw_dates >= "1998-10-01") & (_eval_raw_dates <= "2008-09-30")
_qobs_eval = _eval_npz["qobs_eval"][:, _eval_window].astype(np.float32)

assert "Q_PRED" in globals(), "Mutable cell must define Q_PRED!"
Q_PRED = np.asarray(Q_PRED, dtype=np.float32)
assert Q_PRED.shape == (len(BASINS), len(DATES)), (
    f"Expected Q_PRED shape {(len(BASINS), len(DATES))}, got {Q_PRED.shape}"
)
assert not np.any(np.isnan(Q_PRED[:, ERA_EVAL_MASK])) and not np.any(np.isinf(Q_PRED[:, ERA_EVAL_MASK])), (
    "Q_PRED contains NaN or Inf on ERA_EVAL_MASK (1999-10-01 to 2003-09-30)!"
)

def _compute_basin_nses(q_pred: np.ndarray, q_true: np.ndarray, mask: np.ndarray) -> np.ndarray:
    nses = []
    for i in range(len(BASINS)):
        yt = q_true[i, mask]
        yp = q_pred[i, mask]
        valid = ~np.isnan(yt)
        if np.sum(valid) < 30:
            continue
        yt_v = yt[valid]
        yp_v = yp[valid]
        denom = np.sum((yt_v - np.mean(yt_v)) ** 2)
        if denom > 1e-12:
            nses.append(1.0 - float(np.sum((yp_v - yt_v) ** 2) / denom))
    return np.asarray(nses, dtype=np.float64)

_nses_eval = _compute_basin_nses(Q_PRED, _qobs_eval, ERA_EVAL_MASK)
_median_nse = float(np.median(_nses_eval))
_mean_nse = float(np.mean(_nses_eval))
_q25, _q75 = float(np.percentile(_nses_eval, 25)), float(np.percentile(_nses_eval, 75))
_neg_count = int(np.sum(_nses_eval < 0.0))

print(f"=== ERA Evaluation Summary ({SPATIAL_REGIME} | {len(_nses_eval)} Basins | 1999-10-01 to 2003-09-30) ===")
print(f"Median NSE (1999-2003): {_median_nse:.5f} | Mean NSE: {_mean_nse:.5f} | IQR: [{_q25:.5f}, {_q75:.5f}] | Basins < 0: {_neg_count}")
print(f"Candidate metric to be maximized: {_median_nse:.6f}")

=== ERA Evaluation Summary (pur_18fold | 531 Basins | 1999-10-01 to 2003-09-30) ===
Median NSE (1999-2003): 0.69805 | Mean NSE: 0.18905 | IQR: [0.48379, 0.78273] | Basins < 0: 52
Candidate metric to be maximized: 0.698054


In [7]:
# End Stage